# NovaCred — Privacy, GDPR Compliance & Governance Analysis

**Role:** Governance Officer  
**Inputs:** `applications_curated_full.csv`, `applications_analysis.csv`, `data_quality_report.csv`  
**Source module:** `src/governance.py`

Covers:
1. PII Inventory & Classification
2. Pseudonymisation Demonstration
3. MongoDB Audit Queries (data quality dimensions)
4. GDPR Gap Analysis
5. EU AI Act Classification
6. Governance Recommendations


## Setup

In [1]:
from pathlib import Path
import sys
import pandas as pd

# Resolve the project root (one level above notebooks/) and add to sys.path
# so the src package can be imported without installing it
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# governance.py is the Governance Officer's own module (src/governance.py)
# privacy and config are reused from the Data Engineer's pipeline
import src.governance as governance
from src import privacy, config

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 200)

# QUALITY_DIR  — where the Data Engineer writes its outputs (read-only for us)
# GOVERNANCE_DIR — where this notebook writes its own governance report artefacts
QUALITY_DIR    = PROJECT_ROOT / 'data' / 'quality'
GOVERNANCE_DIR = PROJECT_ROOT / 'reports' / 'Governance'
QUALITY_DIR.mkdir(parents=True, exist_ok=True)
GOVERNANCE_DIR.mkdir(parents=True, exist_ok=True)



## 1. Load Data

We load the outputs produced by the Data Engineer pipeline:
- **`applications_curated_full.csv`** — one row per source JSON record, including all raw and cleaned columns, PII still present (for internal audit use only)
- **`applications_analysis.csv`** — one canonical row per application, with all direct PII removed and replaced by pseudonymous identifiers
- **`data_quality_report.csv`** — the quality issue registry produced by the data quality notebook


In [2]:
# curated_df   — one row per source JSON record; retains raw + cleaned columns
#                 and all PII fields (used here for governance auditing only)
# analysis_df   — one canonical row per application with all direct PII removed;
#                 safe for downstream modelling and fairness analysis
# quality_report — issue registry produced by the Data Engineer's notebook 01

curated_df  = pd.read_csv(PROJECT_ROOT / 'data' / 'curated' / 'applications_curated_full.csv',
                          low_memory=False)
analysis_df = pd.read_csv(PROJECT_ROOT / 'data' / 'curated' / 'applications_analysis.csv',
                          # Keep zip codes and pseudo IDs as strings to preserve
                          # leading zeros and avoid silent numeric coercion
                          dtype={'clean_zip_code': str, 'applicant_pseudo_id': str})
quality_report = pd.read_csv(PROJECT_ROOT / 'data' / 'quality' / 'data_quality_report.csv')

print(f'Curated dataset  : {curated_df.shape[0]:,} rows x {curated_df.shape[1]} columns')
print(f'Analysis dataset : {analysis_df.shape[0]:,} rows x {analysis_df.shape[1]} columns')
print(f'Quality report   : {quality_report.shape[0]} issues logged')

Curated dataset  : 502 rows x 46 columns
Analysis dataset : 500 rows x 16 columns
Quality report   : 33 issues logged


## 2. PII Inventory & Classification

Under GDPR Art. 30, controllers must maintain Records of Processing Activities (ROPA) that include the categories of personal data processed. The table below is our PII inventory: it classifies each field by sensitivity, maps it to the applicable GDPR article, assesses the risk level, and proposes a control.

Fields are classified into three tiers:
- **Direct PII** — uniquely identifies an individual on its own (name, SSN, email)
- **Quasi-PII** — identifies when combined with other fields (gender, zip code)
- **Behavioural data** — reveals patterns that may be used for profiling

In [3]:
# build_pii_catalogue() returns the static field inventory defined in governance.py
# covering all fields classified as Direct PII, Quasi-PII, Behavioural data,
# or Decision output, with GDPR article mappings and recommended controls
pii_catalogue = governance.build_pii_catalogue()
pii_catalogue

,field_path,classification,gdpr_category,gdpr_article,risk,recommendation
0,applicant_info.full_name,Direct PII,Personal data,"Art. 4(1), Art. 5(1)(c)",High,Pseudonymise or redact; retain only in secure audit log
1,applicant_info.email,Direct PII,Personal data,"Art. 4(1), Art. 5(1)(c)",High,Pseudonymise; mask domain for analytics
2,applicant_info.ssn,Direct PII,Personal data (national identifier equivalent),"Art. 4(1), Art. 5(1)(c), Art. 32",Critical,Encrypt at rest; pseudonymise for all processing beyond identity verification
3,applicant_info.ip_address,Direct PII,Online identifier — personal data per GDPR recital 30,"Art. 4(1), Art. 5(1)(c)",High,Retain only for fraud/security purposes; apply storage limitation
4,applicant_info.date_of_birth,Direct PII,Personal data,"Art. 4(1), Art. 5(1)(c)",High,Convert to age band for analytics; retain raw only in secure store
5,applicant_info.gender,Quasi-PII,Personal data,Art. 5(1)(c),Medium,Retain for fairness monitoring only; document lawful basis
6,applicant_info.zip_code,Quasi-PII,Personal data (location proxy),Art. 5(1)(c),Medium,Monitor as potential proxy variable for protected characteristics
7,spending_behavior[].category + amount,Behavioural data,Personal data — behavioural profiling,"Art. 5(1)(b), Art. 22",High,Document purpose limitation; assess whether profiling triggers Art. 22
8,decision.loan_approved + interest_rate,Decision output,Automated decision affecting data subject,"Art. 22, Art. 13",Critical,Provide explanation mechanism; ensure human oversight for adverse decisions


### Key finding
The dataset contains **5 direct PII fields**, including a Social Security Number stored in plain text. Under GDPR Art. 25 (data protection by design), sensitive identifiers must be protected by default — their presence in a raw analytical dataset without encryption or pseudonymisation is a compliance violation.

In [4]:
# Save the PII inventory as governance evidence
# GOVERNANCE_DIR is already defined and created in the Setup cell
pii_catalogue.to_csv(GOVERNANCE_DIR / 'pii_governance_evidence.csv', index=False)
print('PII governance evidence saved to reports/Governance/pii_governance_evidence.csv')


PII governance evidence saved to reports/Governance/pii_governance_evidence.csv


## 3. Pseudonymisation Demonstration

GDPR Art. 25 requires **data protection by design and by default**: personal data should be pseudonymised as early as possible in the processing pipeline. Our Data Engineer has implemented a deterministic pseudonymisation function in `src/privacy.py` using a salted SHA-256 hash.

This section demonstrates the transformation and documents the design choices.

In [5]:
# pseudonymisation_evidence() builds a before/after comparison table:
# which PII fields exist in the curated layer, which are removed in the
# analysis layer, and what derived/pseudonymised replacement was created
pseudo_evidence = governance.pseudonymisation_evidence(curated_df, analysis_df)
pseudo_evidence

,field,raw_field_name,in_curated_layer,in_analysis_layer,treatment,gdpr_principle
0,Full name,raw_applicant_full_name,True,False,Removed from analysis dataset,Data minimisation (Art. 5(1)(c))
1,Email address,raw_applicant_email,True,False,Removed from analysis dataset,Data minimisation (Art. 5(1)(c))
2,Social Security Number,raw_applicant_ssn,True,False,Removed from analysis dataset,Data minimisation (Art. 5(1)(c))
3,IP address,raw_applicant_ip_address,True,False,Removed from analysis dataset,Data minimisation (Art. 5(1)(c))
4,Date of birth,raw_applicant_date_of_birth,True,False,Removed from analysis dataset,Data minimisation (Art. 5(1)(c))
5,SHA-256 pseudonym (salt + SSN/email/name+DOB+zip),→ applicant_pseudo_id,False,True,Derived / pseudonymised replacement,Privacy by design (Art. 25)
6,Age band (coarse — not exact DOB),→ age_band,False,True,Derived / pseudonymised replacement,Privacy by design (Art. 25)
7,Gender (kept for fairness monitoring only),→ clean_gender,False,True,Derived / pseudonymised replacement,Privacy by design (Art. 25)


### How the pseudonymisation works

The `assign_applicant_pseudo_id()` function applies a **priority-based fallback strategy**:

| Priority | Seed | Used when |
|---|---|---|
| 1 | `ssn:{ssn_value}` | SSN is present |
| 2 | `email:{email_value}` | SSN missing, email present |
| 3 | `name_dob_zip:{name}|{dob}|{zip}` | SSN and email missing |
| 4 | `application:{id}|row:{row_id}` | All above missing |

Each seed is hashed as: `SHA-256(salt + "|" + seed)` where the salt is `"novacred_static_salt_v1"`.

**GDPR justification:** Pseudonymisation is explicitly recognised in GDPR recital 26 and Art. 25 as a technical measure that reduces risk. The hash is deterministic (same person = same pseudonym) enabling longitudinal analysis without exposing direct identifiers.

**Limitation:** A static salt means re-identification is possible if the salt is compromised. A production system should use a rotating salt stored in a key management service (e.g., AWS KMS).


In [6]:
# safe_preview_df() masks PII values for safe display in the notebook output:
# names -> [REDACTED_NAME], SSNs -> ***-**-XXXX, emails keep domain, IPs -> [REDACTED_IP]
print('=== Curated layer: PII masked for safe display ===')
safe_preview = privacy.safe_preview_df(
    curated_df, pii_columns=config.DIRECT_PII_COLUMNS, n=5)
display(safe_preview[['application_id', 'raw_applicant_full_name', 'raw_applicant_ssn',
                       'raw_applicant_email', 'raw_applicant_ip_address']].head())

# The analysis layer has no PII columns at all — only the pseudonym and age band
print('\n=== Analysis layer: direct PII removed, pseudonym and age band present ===')
display(analysis_df[['application_id', 'applicant_pseudo_id', 'age_band',
                      'clean_gender', 'pseudo_id_source']].head())

=== Curated layer: PII masked for safe display ===


,application_id,raw_applicant_full_name,raw_applicant_ssn,raw_applicant_email,raw_applicant_ip_address
0,app_200,[REDACTED_NAME],***-**-4340,j***@hotmail.com,[REDACTED_IP]
1,app_037,[REDACTED_NAME],***-**-4784,b***@yahoo.com,[REDACTED_IP]
2,app_215,[REDACTED_NAME],***-**-5178,s***@mail.com,[REDACTED_IP]
3,app_024,[REDACTED_NAME],***-**-1833,t***@protonmail.com,[REDACTED_IP]
4,app_184,[REDACTED_NAME],***-**-2475,b***@aol.com,[REDACTED_IP]



=== Analysis layer: direct PII removed, pseudonym and age band present ===


,application_id,applicant_pseudo_id,age_band,clean_gender,pseudo_id_source
0,app_001,fc4fb76803a008529455aa4130a4c9f4a5f72f06f7ad43b35ac90085631b4a4f,NaN,NaN,email_fallback
1,app_002,7fa4238022da5aed441f8c48a907a8f3cbe88186049c8e6a0980d8854972132f,25-34,Male,ssn
2,app_003,e626311f310f7fb80415229777be761b877b33d42ddcc18ba1f82491890f598b,35-44,Female,ssn
3,app_004,417094dc0567f442dbc01b8d7007ea237a13ba958b0d77efc27f9b35f2aeab4a,25-34,Female,ssn
4,app_005,61c112b16ecefe0dab71d98f848daab97cb0019083597a3e3358cd468d0c9fa0,65+,Female,ssn


In [7]:
# pseudo_id_source records which field was used as the hash seed:
# 'ssn'                  = strongest uniqueness guarantee
# 'email_fallback'        = weaker — email can be shared or reused
# 'name_dob_zip_fallback' = weakest — name collisions are possible
# 'application_id_fallback' = no personal link at all
print('Pseudo ID source distribution:')
print(analysis_df['pseudo_id_source'].value_counts())
print()
ssn_count      = (analysis_df['pseudo_id_source'] == 'ssn').sum()
fallback_count = (analysis_df['pseudo_id_source'] != 'ssn').sum()
print(f'Records using SSN as primary key : {ssn_count:,}')
print(f'Records using fallback strategy  : {fallback_count:,}')
print()
print('Records using a fallback indicate missing SSNs: a completeness issue')
print('that also weakens the uniqueness guarantee of the pseudonym.')

Pseudo ID source distribution:
pseudo_id_source
ssn                      495
name_dob_zip_fallback      4
email_fallback             1
Name: count, dtype: int64

Records using SSN as primary key : 495
Records using fallback strategy  : 5

Records using a fallback indicate missing SSNs: a completeness issue
that also weakens the uniqueness guarantee of the pseudonym.


## 4. Audit Queries — Data Quality Dimensions


The following queries replicate the MongoDB aggregation patterns from the course audit framework, applied to the NovaCred dataset. Each query targets a specific data quality dimension and maps to a GDPR or AI Act governance obligation.


| Dimension | Governance Issue |
|---|---|
|  Uniqueness | Duplicate SSN — identity integrity |
|  Consistency | Gender encoding inconsistency |
| Completeness | Missing consent / governance fields |
|  Validity | Invalid financial values |
|  Fairness | AI Act disparate impact |

### Query 1 — Uniqueness: Duplicate SSNs

**Governance implication:** An SSN should uniquely identify one person. Duplicate SSNs across different application IDs may indicate data entry errors, identity fraud, or a system integrity failure. Under GDPR Art. 5(1)(d), personal data must be accurate and kept up to date.

In [8]:
# curated_df (raw layer) is used deliberately — this is a source integrity check;
# the cleaning pipeline flags duplicates but does not remove them
dup_ssns = governance.mongo_query_1_duplicate_ssns(curated_df)
print(f'SSN values appearing more than once : {len(dup_ssns)}')
if len(dup_ssns) > 0:
    print(f'Total records affected             : {dup_ssns["count"].sum()}')
    display(dup_ssns.head(10))
else:
    print('No duplicate SSNs found at the application level.')
    print('Note: The data quality notebook checks SSNs shared across different application IDs.')

SSN values appearing more than once : 3
Total records affected             : 6


,ssn,count,application_ids
0,652-70-5530,2,app_042|app_042
1,780-24-9300,2,app_088|app_016
2,937-72-8731,2,app_101|app_234


**Finding:** 3 SSN values appear across multiple application IDs, affecting 6 records in total.
SSNs 652-70-5530, 780-24-9300, and 937-72-8731 each appear in 2 distinct applications.
This indicates either data entry errors or potential identity fraud — both represent a failure
of data accuracy under **GDPR Art. 5(1)(d)**. NovaCred cannot determine which application
record is authoritative without manual investigation, and any credit decision made on a
duplicated identity record is legally unreliable.

### Query 2 — Consistency: Gender Encoding

**Governance implication:** Inconsistent categorical encodings cause analytical errors and make fairness monitoring unreliable. If gender is used in fairness audits (required under the AI Act), inconsistent values directly undermine the validity of those audits.

In [9]:
# curated_df is used deliberately — canonical clean values would hide the problem;
# we need to expose what the source system actually sent
gender_consistency = governance.mongo_query_2_gender_consistency(curated_df)
print('Distinct raw gender values found in the dataset:')
display(gender_consistency)
print()
# Identify any value that is not already in canonical form or a recognised null
non_canonical = gender_consistency[
    ~gender_consistency['gender_value'].isin(['Male', 'Female', '[NULL]'])
]
print(f'Non-canonical values requiring normalisation: {len(non_canonical)}')
print('These are cleaned by the Data Engineer pipeline to canonical Male/Female form.')


Distinct raw gender values found in the dataset:


,gender_value,count
0,Male,195
1,Female,193
2,F,58
3,M,53
4,[NULL],3



Non-canonical values requiring normalisation: 2
These are cleaned by the Data Engineer pipeline to canonical Male/Female form.


**Finding:** 5 distinct raw gender values were found: `Male` (195), `Female` (193), `F` (58),
`M` (53), and `[NULL]` (3). This means 111 records (22% of the dataset) used non-canonical
abbreviations, and 3 records have no gender value at all. The Data Engineer's pipeline
normalises these downstream, but the root cause — no input validation at the API layer —
remains unaddressed. Since gender is the protected characteristic used in the AI Act
fairness audit, inconsistent source encoding directly undermines the reliability of the
DI ratio. **GOV-009** recommends enforcing a controlled vocabulary at intake.

### Query 3 — Completeness: Missing Governance Fields (GDPR Gaps)

**Governance implication:** The absence of governance metadata is not merely a data quality issue — it is a **legal compliance failure**. Without consent tracking, NovaCred cannot demonstrate a lawful basis for processing. Without a retention deadline, data may be held indefinitely in violation of GDPR Art. 5(1)(e).

In [10]:
# Pandas equivalent of MongoDB: $match {field: {$exists: false}} + $count
# for each governance metadata field expected by GDPR but absent from the dataset
consent_result = governance.mongo_query_3_missing_consent(curated_df)
print(f'Total records: {consent_result["total_records"]:,}')
print()
# ABSENT  = field does not exist in the schema at all (most severe)
# EMPTY   = field column exists but has no populated values
# OK      = field is present and populated
for field, info in consent_result['fields'].items():
    if info['status'] == 'ABSENT':
        icon = 'CRITICAL'
    elif info['status'] == 'EMPTY':
        icon = 'WARNING '
    else:
        icon = 'OK      '
    print(f'[{icon}] {field:25s} | {info["status"]:20s} | '
          f'Missing: {info["missing_count"]:,} ({info["missing_percent"]}%) | {info["gdpr_reference"]}')


Total records: 502

[CRITICAL] consent_timestamp         | ABSENT               | Missing: 502 (100.0%) | Art. 6 / Art. 7 — Lawful basis / Consent
[CRITICAL] retention_until           | ABSENT               | Missing: 502 (100.0%) | Art. 5(1)(e) — Storage limitation
[CRITICAL] data_source               | ABSENT               | Missing: 502 (100.0%) | Art. 14 — Transparency
[CRITICAL] processing_purpose        | ABSENT               | Missing: 502 (100.0%) | Art. 5(1)(b) — Purpose limitation


**Finding:** All four governance metadata fields are entirely absent from the dataset —
100% missing across all 502 records. This is the most severe compliance finding in this
notebook:

- `consent_timestamp` — NovaCred cannot prove any applicant consented to processing (**Art. 6 / Art. 7**)
- `retention_until` — data may be held indefinitely with no deletion trigger (**Art. 5(1)(e)**)
- `data_source` — no record of where applicant data was obtained (**Art. 14**)
- `processing_purpose` — no documented purpose limits downstream use (**Art. 5(1)(b)**)

Unlike data quality issues which can be remediated in the pipeline, these gaps cannot be
fixed after the fact — they require architectural changes to the intake system before any
new data is collected.

### Query 4 — Validity: Invalid Financial Values

**Governance implication:** Under GDPR Art. 5(1)(d), data used in automated decisions must be accurate. Under EU AI Act Art. 10, high-risk AI training data must meet defined quality criteria, including validity. Invalid inputs directly corrupt the fairness of credit decisions.

In [11]:
# Checks four validity rules — all cases where a value is mathematically impossible:
# negative income, DTI ratio > 1, negative credit history, negative savings balance
validity_result = governance.mongo_query_4_validity(curated_df)
print('Validity issues in financial fields:')
for check, info in validity_result.items():
    # FAIL = at least one record violates this rule
    # PASS = all records satisfy this rule
    flag = 'FAIL' if info['count'] > 0 else 'PASS'
    print(f'  [{flag}] {check:30s}: {info["count"]:>4} records '
          f'({info["percent"]}%) - {info["rule"]}')

Validity issues in financial fields:
  [PASS] negative_income               :    0 records (0.0%) - Validity — income cannot be negative
  [FAIL] dti_out_of_range              :    1 records (0.2%) - Validity — debt-to-income must be in [0, 1]
  [FAIL] negative_credit_history       :    2 records (0.4%) - Validity — credit history months cannot be negative
  [FAIL] negative_savings              :    1 records (0.2%) - Validity — savings balance cannot be negative


**Finding:** 4 validity failures detected across financial fields:

- `dti_out_of_range`: 1 record (0.2%) — debt-to-income ratio exceeds 1.0, which is
  mathematically impossible and indicates a data entry or system error
- `negative_credit_history`: 2 records (0.4%) — negative months of credit history is
  invalid and was nullified by the cleaning pipeline
- `negative_savings`: 1 record (0.2%) — negative savings balance was nullified

While the absolute counts are low, the governance implication is disproportionate: under
**EU AI Act Art. 10**, high-risk AI training and inference data must meet defined quality
criteria. Invalid financial inputs fed into the credit model produce unreliable outputs,
and any adverse decision made on a record with a nullified field lacks a sound factual basis.
The Data Engineer's pipeline flags and nullifies these values — but they should be rejected
at source.

### Query 5 — Fairness: Approval Rate by Gender (AI Act)

**Governance implication:** Under EU AI Act Art. 10, high-risk AI systems must be tested for bias. The Data Scientist's notebook computes the full Disparate Impact Ratio on clean data (DI = 0.77, below the 0.80 four-fifths threshold). This raw query provides the unprocessed view and confirms the bias signal exists in the source data before any normalisation.

In [12]:
# Uses curated_df (raw layer) to confirm the bias signal exists before normalisation,
# ruling out the possibility that the gap is an artefact of the cleaning pipeline
bias_query = governance.mongo_query_5_bias_approval_rate(curated_df)
print('Approval rate by raw gender value (pre-normalisation):')
display(bias_query)
print()
print('Note: Non-canonical encodings (M, F) are normalised by the pipeline.')
print('The Data Scientist notebook computes the definitive DI ratio on clean data.')
print('DI = 0.77 < 0.80 - four-fifths rule BREACHED (see 02-bias-analysis.ipynb).')

Approval rate by raw gender value (pre-normalisation):


,gender_raw,total,approved,approval_rate
0,Male,195,131,0.6718
1,[NULL],3,2,0.6667
2,M,53,32,0.6038
3,Female,193,101,0.5233
4,F,58,26,0.4483



Note: Non-canonical encodings (M, F) are normalised by the pipeline.
The Data Scientist notebook computes the definitive DI ratio on clean data.
DI = 0.77 < 0.80 - four-fifths rule BREACHED (see 02-bias-analysis.ipynb).


**Finding:** Even in the raw pre-normalisation data, a clear approval rate gap is visible
by gender. Combining canonical and abbreviated encodings: Male + M achieve ~65% approval
(131+32 approved out of 195+53), while Female + F achieve only ~51% approval (101+26
approved out of 193+58). The 3 NULL-gender records approve at 67%.

This confirms the bias signal is present at source and is not an artefact of the cleaning
pipeline. The Data Scientist's notebook computes the definitive Disparate Impact Ratio on
the normalised analysis dataset: **DI = 0.77**, which is below the four-fifths threshold
of 0.80, confirming potential disparate impact under **EU AI Act Art. 10**. Remediation
is captured in **GOV-006**.

## 5. GDPR Gap Analysis

A GDPR gap analysis identifies the delta between current data processing practices and the requirements of the GDPR. Each gap below was identified by auditing the dataset schema against GDPR principles and cross-referencing with the findings from the MongoDB queries above.


In [13]:
# build_gdpr_gap_report() checks curated_df for the presence of each expected
# governance field and returns a structured report sorted by severity
gap_report = governance.build_gdpr_gap_report(curated_df)

print('=== Gap Summary by Severity ===')
display(governance.gdpr_gap_summary(gap_report))  # compact count by severity level
print()
print('=== Full GDPR Gap Report ===')
display(gap_report[['gap_id', 'gap_name', 'status', 'gdpr_article', 'severity']])

=== Gap Summary by Severity ===


,Severity,Number of Gaps
0,Critical,3
1,High,4
2,Medium,1
3,Low,0



=== Full GDPR Gap Report ===


,gap_id,gap_name,status,gdpr_article,severity
0,GAP-001,Missing consent timestamp,Absent from dataset,"Art. 6, Art. 7",Critical
1,GAP-005,No audit trail for automated decisions,Absent from dataset,"Art. 22, Art. 13",Critical
2,GAP-006,SSN stored unencrypted,Absent from dataset,"Art. 25, Art. 32",Critical
3,GAP-002,Missing data retention policy,Absent from dataset,Art. 5(1)(e),High
4,GAP-004,Missing processing purpose field,Absent from dataset,Art. 5(1)(b),High
5,GAP-003,Missing data source / transparency field,Absent from dataset,Art. 14,High
6,GAP-007,No human oversight documentation,Absent from dataset,"Art. 22, Art. 14 (AI Act)",High
7,GAP-008,Sensitive behavioural data collected without explicit purpose,Absent from dataset,"Art. 5(1)(b), Art. 22",Medium


In [14]:
# GOVERNANCE_DIR is already defined and created in the Setup cell
gap_report.to_csv(GOVERNANCE_DIR / 'gdpr_gap_report.csv', index=False)
print('GDPR gap report saved to reports/Governance/gdpr_gap_report.csv')

GDPR gap report saved to reports/Governance/gdpr_gap_report.csv


### Critical Gaps — Detailed View

The two most severe findings are:

**GAP-001 — Missing consent timestamp (Art. 6 / Art. 7)**  
There is no `consent_timestamp` field in the dataset. NovaCred cannot prove that any of the 500+ applicants consented to having their data processed. This is potentially the most significant legal exposure in the entire dataset.

**GAP-006 — SSN stored in plain text (Art. 25 / Art. 32)**  
Social Security Numbers — among the most sensitive personal identifiers — are stored without encryption or pseudonymisation in the raw layer. A data breach would trigger mandatory regulatory notification under GDPR Art. 33 and potentially severe fines.


In [15]:
# Filter to Critical-severity gaps only and print the full detail for each
# Description explains the legal exposure; Recommendation proposes the fix
critical = gap_report[gap_report['severity'] == 'Critical']
for _, row in critical.iterrows():
    print('=' * 70)
    print(f'[{row["gap_id"]}] {row["gap_name"]}')
    print(f'GDPR Reference : {row["gdpr_article"]}')
    print(f'Status         : {row["status"]}')
    print(f'\nDescription:')
    print(f'  {row["description"]}')
    print(f'\nRecommendation:')
    print(f'  {row["recommendation"]}')
    print()


[GAP-001] Missing consent timestamp
GDPR Reference : Art. 6, Art. 7
Status         : Absent from dataset

Description:
  No record of when or whether the applicant provided consent for data processing. Without this, NovaCred cannot demonstrate a lawful basis for processing personal data.

Recommendation:
  Implement a consent capture mechanism at application intake. Log timestamp, consent version, and channel. Store in an immutable audit log.

[GAP-005] No audit trail for automated decisions
GDPR Reference : Art. 22, Art. 13
Status         : Absent from dataset

Description:
  Credit decisions appear to be fully automated with no human review record and no explanation of the factors that drove the outcome. Art. 22 grants data subjects the right not to be subject to solely automated decisions with significant effects, unless specific conditions are met.

Recommendation:
  Log the model version, feature weights, and decision rationale for every application. Implement a human-in-the-loop 

### Evidence: Algorithm Risk Score as Dominant Rejection Reason (GAP-005)

In [16]:
# Examine the dominant rejection reason as concrete evidence for GAP-005 (missing audit trail for automated decisions)
# We use analysis_df here — rejection reasons contain no PII and this is an outcome analysis rather than a raw data audit
rejected      = analysis_df[analysis_df['clean_loan_approved'] == False]
reason_counts = rejected['clean_rejection_reason'].value_counts()

# Calculate the share of rejections citing the opaque algorithm score
algo_count = reason_counts.get('algorithm_risk_score', 0)
algo_pct   = algo_count / len(rejected) * 100

print('All rejection reasons:')
print(reason_counts.to_string())
print()
print(f"Rejections citing 'algorithm_risk_score' : {algo_count} ({algo_pct:.1f}%)")
print("'algorithm_risk_score' provides no meaningful explanation to the applicant")
print("- GDPR Art. 22 violation and EU AI Act Art. 13 transparency failure.")

All rejection reasons:
clean_rejection_reason
algorithm_risk_score           169
insufficient_credit_history     23
high_dti_ratio                  12
low_income                       4

Rejections citing 'algorithm_risk_score' : 169 (81.2%)
'algorithm_risk_score' provides no meaningful explanation to the applicant
- GDPR Art. 22 violation and EU AI Act Art. 13 transparency failure.


## 6. EU AI Act Classification

The EU AI Act, which entered into force in August 2024, classifies AI systems by risk level. The obligations that apply depend entirely on this classification. NovaCred's credit scoring system falls squarely within the **high-risk** category.

In [17]:
# build_ai_act_classification() returns NovaCred's EU AI Act risk classification
# and the list of legal obligations that apply to high-risk AI systems
ai_act = governance.build_ai_act_classification()

print(f'System      : {ai_act["system_name"]}')
print(f'Risk Level  : {ai_act["risk_level"]}')
print(f'Legal Basis : {ai_act["legal_basis"]}')
print()
# Each row in the obligations table is one Article with its description
obligations_df = governance.build_ai_act_summary_df(ai_act)
display(obligations_df)

System      : NovaCred Automated Credit Scoring System
Risk Level  : HIGH-RISK
Legal Basis : EU AI Act Annex III, Point 5(b) — Creditworthiness assessment and credit scoring



,article,obligation,description
0,Art. 9,Risk management system,Implement and maintain a risk management system throughout the entire lifecycle of the AI system. Identify and analy...
1,Art. 10,Data governance,"Training, validation, and testing data must meet quality criteria. Data must be examined for biases. Data gaps and s..."
2,Art. 13,Transparency,The AI system must be sufficiently transparent to enable deployers to interpret its output. A technical document mus...
3,Art. 14,Human oversight,"High-risk AI systems must be designed to allow effective human oversight. Humans must be able to intervene, override..."
4,Art. 26,Deployer obligations,"NovaCred (as deployer) must ensure the system is used in accordance with instructions, monitor operation, and inform..."
5,Art. 72,EU database registration,High-risk AI systems in scope of Annex III must be registered in the EU database before being placed on the market o...
6,Art. 35 (GDPR),DPIA required,Automated credit decisions using profiling likely require a Data Protection Impact Assessment under GDPR Art. 35 bef...


In [18]:
# List the immediate gaps identified against AI Act obligations
# These are findings from this notebook mapped to specific AI Act articles
print('Immediate compliance gaps identified:')
for idx, gap in enumerate(ai_act['immediate_gaps'], 1):
    print(f'  {idx}. {gap}')


Immediate compliance gaps identified:
  1. No human oversight mechanism documented in the dataset
  2. No audit trail linking decisions to model version or feature inputs
  3. No DPIA evidence present
  4. No EU AI Act registration recorded
  5. Fairness testing shows DI = 0.77 (below 0.80 threshold) — potential Art. 10 violation


### Why High-Risk?

EU AI Act **Annex III, Point 5(b)** explicitly lists:
> *'AI systems intended to be used for creditworthiness assessment or credit scoring of natural persons'*

NovaCred satisfies all three conditions: it is an ML system, it assesses creditworthiness, and it acts on individual natural persons.

The combination of GDPR Art. 22 (automated decision-making) and EU AI Act high-risk obligations creates a **dual compliance requirement** — both regimes apply simultaneously and reinforce each other.

## 7. Governance Recommendations

The following recommendations are derived from the GDPR gap analysis, the AI Act classification, and the bias findings from the Data Scientist's notebook. They are prioritised by legal severity and implementation urgency.

In [19]:
# build_governance_recommendations() returns 10 prioritised controls
# derived from the GDPR gap analysis, AI Act classification, and bias findings
# Each row has a legal reference, effort estimate, and responsible role
recommendations = governance.build_governance_recommendations()
display(recommendations[[
    'priority', 'control_id', 'category', 'title', 'legal_ref', 'effort', 'responsible'
]])


,priority,control_id,category,title,legal_ref,effort,responsible
0,1,GOV-001,Legal Compliance,Implement consent capture and tracking,"GDPR Art. 6, Art. 7",Medium,Engineering + Legal
1,2,GOV-002,Legal Compliance,Define and enforce data retention policy,"GDPR Art. 5(1)(e), Art. 30",Medium,Data Engineering + DPO
2,3,GOV-003,Security,Encrypt and pseudonymise SSNs,"GDPR Art. 25, Art. 32",Low,Data Engineering
3,4,GOV-004,AI Act / Automated Decisions,Implement human oversight mechanism,"GDPR Art. 22, EU AI Act Art. 14",High,Product + Operations
4,5,GOV-005,AI Act / Automated Decisions,Create decision audit trail,"GDPR Art. 22, EU AI Act Art. 13",Medium,Data Science + Engineering
5,6,GOV-006,Fairness,Address gender disparate impact (DI = 0.77),"EU AI Act Art. 10, GDPR Art. 22",High,Data Science + Legal
6,7,GOV-007,Transparency,Add data source and processing purpose fields,"GDPR Art. 5(1)(b), Art. 14",Low,Engineering + Legal
7,8,GOV-008,AI Act,Conduct DPIA and register with EU AI Act database,"GDPR Art. 35, EU AI Act Art. 72",High,DPO + Legal + Management
8,9,GOV-009,Data Quality,Standardise gender encoding at source,GDPR Art. 5(1)(d) — Accuracy,Low,Data Engineering
9,10,GOV-010,Privacy by Design,Apply data minimisation to spending behaviour data,"GDPR Art. 5(1)(c), Art. 25",Medium,Data Science + DPO


In [20]:
recommendations.to_csv(GOVERNANCE_DIR / 'governance_recommendations.csv', index=False)
print("Governance recommendations saved to reports/govenance/governance_recommendations.csv")

Governance recommendations saved to reports/govenance/governance_recommendations.csv


### Top 3 Immediate Actions

| Priority | Control | Action | Why Urgent |
|---|---|---|---|
| 1 | GOV-001 | Implement consent tracking | Cannot prove lawful basis — all processing at legal risk |
| 2 | GOV-003 | Encrypt & pseudonymise SSNs | Critical breach risk; pipeline already ready |
| 3 | GOV-004 | Add human oversight mechanism | Required by GDPR Art. 22 and EU AI Act Art. 14 |

## 8. Governance Summary

In [21]:
print('=' * 60)
print('NOVACRED — GOVERNANCE AUDIT SUMMARY')
print('=' * 60)

total_gaps    = len(gap_report)
critical_gaps = (gap_report['severity'] == 'Critical').sum()
high_gaps     = (gap_report['severity'] == 'High').sum()

print(f'\nGDPR Gaps identified  : {total_gaps}')
print(f'  Critical            : {critical_gaps}')
print(f'  High                : {high_gaps}')
print(f'\nPII fields catalogued : {len(pii_catalogue)}')
print(f'  Direct PII          : {(pii_catalogue["classification"] == "Direct PII").sum()}')
print(f'\nAI Act risk level     : HIGH-RISK (Annex III Section 5(b))')
print(f'AI Act obligations    : {len(ai_act["obligations"])}')
print(f'Immediate AI Act gaps : {len(ai_act["immediate_gaps"])}')
print(f'\nGovernance controls   : {len(recommendations)} recommended')
print()
print('Key finding: NovaCred processes highly sensitive personal data in an')
print('automated high-risk AI system with critical GDPR compliance gaps.')
print('Immediate remediation is required before any regulatory review.')

NOVACRED — GOVERNANCE AUDIT SUMMARY

GDPR Gaps identified  : 8
  Critical            : 3
  High                : 4

PII fields catalogued : 9
  Direct PII          : 5

AI Act risk level     : HIGH-RISK (Annex III Section 5(b))
AI Act obligations    : 7
Immediate AI Act gaps : 5

Governance controls   : 10 recommended

Key finding: NovaCred processes highly sensitive personal data in an
automated high-risk AI system with critical GDPR compliance gaps.
Immediate remediation is required before any regulatory review.
